<a href="https://colab.research.google.com/github/girish17work/Data-Analysis/blob/main/Unknown_Dataset_6/Unknown_Dataset_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [47]:
DATASET_URL='https://drive.google.com/uc?export=download&id=1A33DnF50NxCkqyInweMSjQm88f3CyqwL'
df = pd.read_csv(DATASET_URL)
df.head()

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Month,WeekDay,Day
0,Jet Airways,Delhi,Banglore,DEL → BOM → COK,20:00,04:25 10 Jun,26h 35m,1 stop,In-flight meal not included,14924,6,Thursday,6.0
1,Jet Airways,Delhi,Cochin,DEL → BOM → COK,16:00,19:00 10 Jun,27h,1 stop,In-flight meal not included,10577,6,Sunday,9.0
2,Jet Airways,Mumbai,Hyderabad,BOM → HYD,19:35,21:05,1h 30m,non-stop,No info,5678,3,Friday,15.0
3,Multiple carriers,Delhi,Banglore,DEL → BOM → COK,18:55,01:30 16 Jun,15h 10m,1 stop,In-flight meal not included,7408,5,Monday,6.0
4,Air India,Delhi,Cochin,DEL → COK,17:10,17:55,8h 20m,non-stop,No info,6724,6,Monday,24.0


In [48]:
print(f'Median of the Flight Ticket Price: {df['Price'].median()}')

Median of the Flight Ticket Price: 8366.0


In [49]:
print(f'Lowest number of flights occurred in {df['Month'].value_counts().idxmin()}')

Lowest number of flights occurred in 4


In [50]:
print(f'Source with highest average ticket price: {df.groupby('Source')['Price'].mean().idxmax()}')

Source with highest average ticket price: Delhi


In [51]:
df['Additional_Info'] = df['Additional_Info'].replace('No Info', 'No info')
print(f'Flights with airline Jet Airways and have No info as additional information: {((df['Airline']=='Jet Airways') & (df['Additional_Info']=='No info')).sum()}')

Flights with airline Jet Airways and have No info as additional information: 2119


In [52]:
def to_minutes(s):
  h = m = 0
  for part in s.split():
    if part.endswith('h'):
      h = int(part[:-1])
    elif part.endswith('m'):
      m = int(part[:-1])
  return h*60+m

df['Duration'] = df['Duration'].apply(to_minutes)
print(f'Median Duration of a Flight: {df['Duration'].median()}')

Median Duration of a Flight: 520.0


In [53]:
def part_of_day(t):
  hour = int(t.split(':')[0])
  if 5 <= hour < 12:
    return 'Morning'
  elif 12 <= hour < 17:
    return 'Afternoon'
  elif 17 <= hour < 20:
    return 'Evening'
  return 'Night'

df['Dep_Time'] = df['Dep_Time'].apply(part_of_day)
df['Arrival_Time'] = df['Arrival_Time'].apply(part_of_day)

In [54]:
print(f'Number of flights started in the Evening and arrived at the destination at Night: {((df['Dep_Time']=='Evening') & (df['Arrival_Time']=='Night')).sum()}')

Number of flights started in the Evening and arrived at the destination at Night: 732


In [55]:
df['WeekDay'] = df['WeekDay'].isin(['Saturday', 'Sunday']).astype(int)
print(df['WeekDay'].sum())

2799


In [56]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Price'])
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f'Number of rows in the feature matrix of the train dataset: {X_train.shape[0]}')

Number of rows in the feature matrix of the train dataset: 7087


In [57]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline

X_train = X_train.drop(columns=['Route'])
X_test = X_test.drop(columns=['Route'])

times = ['Morning', 'Afternoon', 'Evening', 'Night']
stops = ['non-stop', '1 stop', '2 stops', '3 stops', '4 stops']

total_stop_pipeline = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oe', OrdinalEncoder(categories=[stops], handle_unknown='use_encoded_value', unknown_value=-1))])

preprocessor = ColumnTransformer([
    ('total_stops', total_stop_pipeline, ['Total_Stops']),
    ('dur', MinMaxScaler(), ['Duration']),
    ('day', SimpleImputer(strategy='median'), ['Day']),
    ('nominal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ['Airline', 'Source', 'Destination', 'Additional_Info']),
    ('dep_time', OrdinalEncoder(categories=[times, times], handle_unknown='use_encoded_value', unknown_value=-1), ['Dep_Time', 'Arrival_Time'])
], remainder='passthrough').set_output(transform='pandas')

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

In [58]:
print(f'Presence of unseen value in the test dataset: {(X_test_t==-1).any().any()}')

Presence of unseen value in the test dataset: True


In [59]:
print(f'Highest value in the transformed test feature matrix: {X_test_t.to_numpy().max()}')

Highest value in the transformed test feature matrix: 27.0


In [60]:
print(f'Mean of the transformed training matrix: {round(X_train_t.to_numpy().mean(), 2)}')

Mean of the transformed training matrix: 3.19


In [61]:
print(f'Highest value of the scaled duration column in the tranformed test dataset: {round(X_test_t['dur__Duration'].max(),2)}')

Highest value of the scaled duration column in the tranformed test dataset: 0.82


In [62]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=1fQgNQ85XoPzC3Pj_Yyq940jwUI_qE7fY'
df = pd.read_csv(DATASET_URL)
df.head()

,0,1,2,3,4,5,6,7,8,9,10,11
0,0.201049,3.0,0.0,1.0,6.0,0.0,0.0,3.0,27.0,5,0,6144
1,0.461538,3.0,3.0,1.0,6.0,0.0,1.0,3.0,12.0,6,0,4778
2,0.059441,2.0,0.0,2.0,6.0,0.0,0.0,1.0,3.0,4,0,3898
3,0.054196,3.0,3.0,0.0,6.0,0.0,0.0,1.0,9.0,5,0,5224
4,0.190559,8.0,2.0,0.0,6.0,1.0,0.0,1.0,12.0,6,0,6356


In [63]:
from sklearn.model_selection import train_test_split

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [64]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

ridge = Ridge(alpha=1, solver='saga', tol=1e-3, random_state=42)
ridge.fit(X_train,y_train)

y_pred = ridge.predict(X_test)

print(f'R2 score on ridge model: {round(r2_score(y_test, y_pred), 3)}')

R2 score on ridge model: 0.413


In [65]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

estimator = SGDRegressor(random_state=42)
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [1e-4, 1e-3, 1e-2, 1e-1, 1],
    'tol': [1e-3, 1e-2]
}

grid_search_cv = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search_cv.fit(X_train, y_train)

best_estimator = grid_search_cv.best_estimator_

print(f'Best alpha: {grid_search_cv.best_params_['alpha']}')

Best alpha: 1


In [66]:
from sklearn.metrics import root_mean_squared_error

y_pred = best_estimator.predict(X_test)

print(f'Root Mean Squared Error: {round(root_mean_squared_error(y_test, y_pred), 2)}')

Root Mean Squared Error: 3580.16


In [67]:
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('pca', PCA()), ('lasso', Lasso())])
param_grid = {
    'pca__n_components': [0.85, 0.9],
    'lasso__alpha': [10, 1, 0.1]
}
grid_search_cv = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, scoring='neg_mean_absolute_error')
grid_search_cv.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('pca', PCA()), ('lasso', Lasso())]),
             n_jobs=-1,
             param_grid={'lasso__alpha': [10, 1, 0.1],
                         'pca__n_components': [0.85, 0.9]},
             scoring='neg_mean_absolute_error')

In [68]:
from sklearn.metrics import mean_absolute_error

y_pred = grid_search_cv.best_estimator_.predict(X_test)

print(f'Mean absolute error on test dataset: {round(mean_absolute_error(y_test, y_pred), 2)}')

Mean absolute error on test dataset: 3630.4


In [72]:
print(f'Fraction of variance explained by the first principal component: {round(grid_search_cv.best_estimator_.named_steps['pca'].explained_variance_ratio_[0], 2)}')

Fraction of variance explained by the first principal component: 0.83


In [73]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

pipeline = Pipeline([('poly', PolynomialFeatures(interaction_only=False, degree=3)), ('lasso', Lasso(alpha=50, warm_start=True))])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print(f'R2 score on the test dataset: {round(r2_score(y_test, y_pred), 2)}')

R2 score on the test dataset: 0.53
